In [ ]:
import pandas as pd
import json
import random
import numpy as np

import matplotlib.pyplot as plt
from PIL import Image

from openai import OpenAI
from pydantic import BaseModel
from typing import Optional

import base64
from tqdm import tqdm
import time
from io import BytesIO
import re
import torch

import unicodedata as ud
from typing import Union

In [ ]:
def save_txt(text: str, path: str):
    with open(path, 'w') as f:
        f.write(text)

def save_json(data: dict, path: str):
    with open(path, 'w') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

def load_json(path: str):
    with open(path, 'r') as f:
        return json.load(f)
    
def load_law_data(path: str):
    raw_law_data = load_json(path)
    return {e["id"]: e["articles"]  for e in raw_law_data}

def save_law_data(law_data: dict, path: str):
    raw_law_data = [{"id": k, "articles": v} for k, v in law_data.items()]
    save_json(raw_law_data, path)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
from transformers import CLIPModel, CLIPProcessor, CLIPConfig
model_id = "zer0int/CLIP-GmP-ViT-L-14"
config = CLIPConfig.from_pretrained(model_id)
processor = CLIPProcessor.from_pretrained(model_id)
embedding_model = CLIPModel.from_pretrained(model_id)

embedding_model = embedding_model.to("cuda")
embedding_model.eval()

count_parameters(embedding_model)

In [ ]:
law_data = load_law_data("data/law_db/complete_preprocessed_vlsp2025_law_with_traffic_signs.json")
law_data
USE_SIGNS_WITH_IMAGES = True

relevant_article_embeddings = []
relevant_article_identities = []
for k, v in law_data.items():
    for article_i, e in tqdm(enumerate(v), total=len(v)):
        if USE_SIGNS_WITH_IMAGES:
            for sign_i, sign in enumerate(e["signs_with_images"]):
                for image_i, image_name in enumerate(sign["image_names"]):
                    relevant_article_identities.append({
                        "law_id": k,
                        "article_index": article_i,
                        "article_id": e["id"],
                        "article": e,
                        "sign_index": sign_i,
                        "image_index": image_i,
                        "sign": sign,
                    })
                    image = Image.open(f"data/law_db/images.fld/{image_name}")
                    inputs = processor(images=image, return_tensors="pt").to("cuda")
                    image_features = embedding_model.get_image_features(**inputs)
                    image_features = image_features.detach().cpu().numpy()
                    relevant_article_embeddings.append(image_features)
        else:
            for i, image_name in enumerate(e["images"]):
                relevant_article_identities.append({
                    "law_id": k,
                    "article_index": article_i,
                    "article_id": e["id"],
                    "article": e,
                    "image_index": i,
                    "images": e["images"],
                })
                image = Image.open(f"data/law_db/images.fld/{image_name}")
                inputs = processor(images=image, return_tensors="pt").to("cuda")
                image_features = embedding_model.get_image_features(**inputs)
                image_features = image_features.detach().cpu().numpy()
                relevant_article_embeddings.append(image_features)

relevant_article_embeddings = np.concatenate(relevant_article_embeddings, axis=0)
relevant_article_embeddings.shape

if USE_SIGNS_WITH_IMAGES:
    np.save('data/law_db/image_only_relevant_article_embeddings_with_traffic_signs.npy', relevant_article_embeddings)
    save_json(relevant_article_identities, 'data/law_db/image_only_relevant_article_identities_with_traffic_signs.json')
else:
    np.save('data/law_db/image_only_relevant_article_embeddings.npy', relevant_article_embeddings)
    save_json(relevant_article_identities, 'data/law_db/image_only_relevant_article_identities.json')

In [ ]:
if USE_SIGNS_WITH_IMAGES:
    relevant_article_embeddings = np.load('data/law_db/image_only_relevant_article_embeddings_with_traffic_signs.npy')
    relevant_article_identities = load_json('data/law_db/image_only_relevant_article_identities_with_traffic_signs.json')
else:
    relevant_article_embeddings = np.load('data/law_db/image_only_relevant_article_embeddings.npy')
    relevant_article_identities = load_json('data/law_db/image_only_relevant_article_identities.json')


In [ ]:
law_data = load_law_data("data/law_db/preprocessed_vlsp2025_law_with_traffic_signs.json")

train_data = load_json("data/train_data/vlsp_2025_train.json")
train_data


In [ ]:
sample = train_data[21]
sample

In [ ]:
image_path = "data/train_data/train_images/" + sample["image_id"] + ".jpg"
print("image_path:", image_path)
image = Image.open(image_path).convert("RGB")
image

In [ ]:
import torch
from torchvision.ops import box_iou


def postprocess_boxes(result, image_size, aspect_ratio_threshold=1.5, area_ratio_threshold=0.005):
    """Remove boxes with ratio (height / width) > threshold or too small"""
    w, h = image_size
    image_area = h * w

    boxes = result[0]['boxes']
    scores = result[0]['scores']
    labels = result[0]['labels']
    text_labels = result[0]['text_labels']

    keep_indices = []
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box
        width = x2 - x1
        height = y2 - y1

        box_area = width * height
        if box_area / image_area < area_ratio_threshold:
            continue

        if height / width <= aspect_ratio_threshold:
            keep_indices.append(i)

    filtered_result = {
        'boxes': boxes[keep_indices],
        'scores': scores[keep_indices],
        'labels': [labels[i] for i in keep_indices],
        'text_labels': [text_labels[i] for i in keep_indices]
    }
    return [filtered_result]


def apply_simple_nms(result, iou_threshold=0.5):
    boxes = result[0]['boxes']
    scores = result[0]['scores']
    labels = result[0]['labels']
    text_labels = result[0]['text_labels']

    if boxes.size(0) == 0:
        return result

    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    ious = box_iou(boxes, boxes)

    keep = []
    suppressed = torch.zeros(len(boxes), dtype=torch.bool, device=boxes.device)

    for i in range(len(boxes)):
        if suppressed[i]:
            continue

        overlap_inds = (ious[i] > iou_threshold).nonzero(as_tuple=False).flatten()

        labeled_inds = [j.item() for j in overlap_inds if text_labels[j] not in ['', None]]

        if labeled_inds:
            selected_idx = max(labeled_inds, key=lambda j: areas[j])
        else:
            selected_idx = overlap_inds[torch.argmax(areas[overlap_inds])].item()

        keep.append(selected_idx)
        suppressed[overlap_inds] = True

    filtered_result = {
        'boxes': boxes[keep],
        'scores': scores[keep],
        'labels': [labels[i] for i in keep],
        'text_labels': [text_labels[i] for i in keep]
    }
    return [filtered_result]

import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection


class GroundingDINObjectDetector:
  def __init__(self):
    self.model_id = "IDEA-Research/grounding-dino-base"
    self.device = "cuda" if torch.cuda.is_available() else "cpu"

    self.processor = AutoProcessor.from_pretrained(self.model_id)
    self.model = AutoModelForZeroShotObjectDetection.from_pretrained(self.model_id).to(self.device)

  def __call__(self, image):
    # all lowercase, each class separated with dot.
    text = "rectangle blue and white sign. circular green and white sign. triangle red and yellow sign. circular red and white traffic sign."

    inputs = self.processor(images=image, text=text, return_tensors="pt").to(self.device)
    with torch.no_grad():
        outputs = self.model(**inputs)

    results = self.processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        box_threshold=0.15,
        text_threshold=0.3,
        target_sizes=[image.size[::-1]]
    )

    results = postprocess_boxes(results, image.size)
    results = apply_simple_nms(results, 0.3)

    return results

from PIL import ImageDraw, ImageFont

def draw_grounding_dino_results(image, results):
    draw = ImageDraw.Draw(image)

    for result in results:
        boxes = result["boxes"]
        scores = result["scores"]
        labels = result["text_labels"]

        for i in range(len(boxes)):
            score = scores[i].item()

            box = boxes[i].tolist()
            label = labels[i]

            x1, y1, x2, y2 = box
            draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
            draw.text((x1, y1 - 10), f"{label} ({score:.2f})", fill="red")

    return image

In [ ]:
object_detector_model = GroundingDINObjectDetector()
object_detector_model.model.eval()
print(count_parameters(object_detector_model.model))
object_detector_model

In [ ]:
def show_similarity_lst(similarity_scores_np: np.array, top_k: int):
    top_k_indices = np.argsort(similarity_scores_np)[-top_k:][::-1]
    top_k_scores = similarity_scores_np[top_k_indices]
    # print(sample["relevant_articles"])
    print("top_k_indices", top_k_indices)
    # print("top_k_scores", top_k_scores)
    for i, (index, score) in enumerate(zip(top_k_indices, top_k_scores)):
        image_index = relevant_article_identities[index]['image_index']
        law_id = relevant_article_identities[index]['law_id']
        article_index = relevant_article_identities[index]['article_index']
        article_id = relevant_article_identities[index]['article_id']

        article = law_data[law_id][article_index]
        # print(i, score, "Relevant article:", , , relevant_article_identities[index]['image_index'])
        image_name = article["images"][image_index]
        image = Image.open(f"data/law_db/images.fld/{image_name}")
        print(i, score, "Relevant article:", law_id, article_id, image_name)
        display(image)

In [ ]:
sample = train_data[21]
sample

In [ ]:
image_path = "data/train_data/train_images/" + sample["image_id"] + ".jpg"
print("image_path:", image_path)
image = Image.open(image_path).convert("RGB")

results = object_detector_model(image)
result_image = draw_grounding_dino_results(image.copy(), results)
display(result_image)

In [ ]:
boxes = results[0]['boxes'].detach().cpu().numpy().tolist()
scores = results[0]['scores'].detach().cpu().numpy().tolist()

TOP_K = 40
labels = results[0]['labels']

similarity_scores_lst = []

for box, score, label in zip(boxes, scores, labels):
    cropped_image = image.crop(box)
    display(cropped_image)

    inputs = processor(images=cropped_image, return_tensors="pt").to("cuda")
    image_features = embedding_model.get_image_features(**inputs)
    image_features = image_features.detach().cpu().numpy()
    similarity_scores = np.dot(relevant_article_embeddings / np.linalg.norm(relevant_article_embeddings, axis=1, keepdims=True), (image_features/np.linalg.norm(image_features)).T)
    similarity_scores_lst.append(similarity_scores)
    print("Detection score", score)

    show_similarity_lst(np.squeeze(similarity_scores, axis=1), TOP_K)
    print("============================================================")

similarity_scores_np = np.concatenate(similarity_scores_lst, axis=1)
similarity_scores_np = np.max(similarity_scores_np, axis=1)

show_similarity_lst(similarity_scores_np, TOP_K)